# Lab 03 - Notebooks Avanzados

**Objetivos:**
- Dominar magic commands
- Usar widgets para parametrización
- Comunicación entre notebooks
- Debugging y logging

## Ejercicio 1: Magic Commands

### Markdown

In [ ]:
# =============================================================================
# MAGIC COMMAND: %md (Markdown)
# =============================================================================
# Objetivo: Renderizar markdown directamente sin crear celda separada
# Útil para: Documentación dinámica generada desde código

# %md es un "magic command" que cambia el comportamiento de la celda
# Todo el contenido después de %md se interpreta como Markdown

%md
# Título usando Magic Command
Este es un **markdown** usando `%md`

- Item 1
- Item 2
- Item 3

# 💡 MAGIC COMMANDS DISPONIBLES EN DATABRICKS:
# - %md: Renderizar Markdown
# - %sql: Ejecutar consultas SQL
# - %python, %scala, %r: Cambiar lenguaje de la celda
# - %sh: Ejecutar comandos shell en el driver node
# - %fs: Comandos del file system (atajos para dbutils.fs)
# - %pip: Instalar paquetes Python en runtime
# - %run: Ejecutar otro notebook

### Crear datos de prueba

In [ ]:
# =============================================================================
# GENERACIÓN DE DATOS SINTÉTICOS DE VENTAS
# =============================================================================
# Objetivo: Crear dataset realista para testear widgets y queries
# Sin dependencias externas (todo en memoria)

# Importaciones
from pyspark.sql.functions import *  # Funciones de transformación
from datetime import datetime, timedelta  # Manejo de fechas
import random  # Generación aleatoria

# Datos maestros (catálogos)
products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"]
regions = ["North", "South", "East", "West"]

# Generar 100 ventas aleatorias
data = []
for i in range(100):
    data.append({
        # ID de venta con formato S0001, S0002, etc.
        "sale_id": f"S{i:04d}",  # :04d = 4 dígitos con padding de ceros
        
        # Fecha aleatoria en los últimos 30 días
        "date": (datetime.now() - timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d"),
        
        # Producto aleatorio del catálogo
        "product": random.choice(products),
        
        # Región aleatoria
        "region": random.choice(regions),
        
        # Monto entre $100 - $2000, con 2 decimales
        "amount": round(random.uniform(100, 2000), 2),
        
        # Cantidad entre 1-5 unidades
        "quantity": random.randint(1, 5)
    })

# 💡 NOTA: Este enfoque es ideal para:
# - Desarrollo y testing (no requiere datos reales)
# - Demos y capacitaciones
# - Unit tests de lógica de negocio
# - Prototipado rápido de pipelines

# Crear DataFrame desde lista de diccionarios
# Spark infiere el schema automáticamente desde las keys/values
df_sales = spark.createDataFrame(data)

# Crear vista temporal para poder usar SQL
# "sales" será el nombre de la tabla en queries SQL
df_sales.createOrReplaceTempView("sales")

print(f"✅ Datos creados: {df_sales.count()} ventas")
display(df_sales.limit(5))  # Mostrar solo 5 filas de ejemplo

### SQL Query

In [ ]:
# =============================================================================
# MAGIC COMMAND: %%sql (SQL Query)
# =============================================================================
# Objetivo: Ejecutar SQL directamente contra temp views
# Ventaja: No necesitas spark.sql("..."), más limpio y legible

# %%sql (con doble %%) convierte TODA la celda a SQL
# Puedes usar TODA la sintaxis de SQL estándar

%%sql
-- Consulta: Top 10 combinaciones Region-Product por ingresos
-- Usamos agregaciones (COUNT, SUM, AVG) y ordenamiento
-- 📊 RESULTADO: Tabla interactiva con opciones de visualización
-- Databricks automáticamente renderiza gráficos (barras, líneas, pie, etc.)

SELECT 
    region,                               -- Región de venta
    product,                              -- Producto vendido
    COUNT(*) as num_sales,                -- Total de ventas (transacciones)
    ROUND(SUM(amount), 2) as total_revenue,  -- Ingresos totales con 2 decimales
    ROUND(AVG(amount), 2) as avg_amount   -- Ticket promedio
FROM sales                                -- Desde la temp view "sales"
GROUP BY region, product                  -- Agrupar por región y producto
ORDER BY total_revenue DESC               -- Ordenar por ingresos (mayor a menor)
LIMIT 10                                  -- Solo top 10 resultados

-- 💡 EQUIVALENTE EN DATAFRAME API:
-- df_sales.groupBy("region", "product") \
--   .agg(
--     count("*").alias("num_sales"),
--     round(sum("amount"), 2).alias("total_revenue"),
--     round(avg("amount"), 2).alias("avg_amount")
--   ) \
--   .orderBy(col("total_revenue").desc()) \
--   .limit(10)

### Shell Commands

In [ ]:
%sh
echo "📁 Listar directorio actual:"
pwd
ls -lh

### File System Commands

In [ ]:
%fs ls /tmp/

## Ejercicio 2: Widgets (Parametrización)

In [ ]:
# =============================================================================
# WIDGETS: PARAMETRIZACIÓN DE NOTEBOOKS
# =============================================================================
# Objetivo: Hacer notebooks interactivos y reutilizables
# Widgets permiten: Ejecutar mismo notebook con diferentes parámetros

# dbutils.widgets es la API de Databricks para crear controles de entrada
# Los widgets aparecen en la parte superior del notebook

# 🔄 CASOS DE USO:
# 1. Jobs parametrizados: Mismo notebook, diferentes parámetros por ejecución
# 2. Dashboards interactivos: Usuarios filtran datos sin modificar código
# 3. Testing: Probar diferentes escenarios fácilmente
# 4. Orchestration: Pasar parámetros entre notebooks

# 💡 TIPOS DE WIDGETS DISPONIBLES:
# - text(): Entrada de texto libre (fechas, nombres, paths)
# - dropdown(): Selección única (estados, tipos, categorías)
# - combobox(): Dropdown con opción de texto custom (flexible)
# - multiselect(): Selección múltiple (filtros, columnas)

# WIDGET 1: Text Input (entrada de texto libre)
dbutils.widgets.text(
    "fecha_inicio",     # Nombre interno del widget (para leer valor)
    "2026-05-01",       # Valor por defecto
    "📅 Fecha Inicio"  # Label visible al usuario
)

# WIDGET 2: Dropdown (selección única de lista cerrada)
dbutils.widgets.dropdown(
    "region",           # Nombre interno
    "All",              # Valor por defecto seleccionado
    ["All", "North", "South", "East", "West"],  # Opciones disponibles
    "🌍 Región"       # Label
)

# WIDGET 3: Combobox (dropdown + texto libre)
dbutils.widgets.combobox(
    "producto",         # Nombre interno
    "All",              # Valor por defecto
    ["All"] + products, # Opciones predefinidas + permite texto custom
    "📦 Producto"      # Label
)

# WIDGET 4: Multiselect (selección múltiple)
dbutils.widgets.multiselect(
    "metricas",                        # Nombre interno
    "amount,quantity",                 # Valores por defecto (separados por coma)
    ["amount", "quantity", "sale_id"], # Opciones disponibles
    "📊 Métricas"                      # Label
)

print("✅ Widgets creados")

In [ ]:
# Leer valores de widgets
fecha_inicio = dbutils.widgets.get("fecha_inicio")
region_filtro = dbutils.widgets.get("region")
producto_filtro = dbutils.widgets.get("producto")
metricas = dbutils.widgets.get("metricas").split(",")

print("📋 Parámetros seleccionados:")
print(f"  Fecha inicio: {fecha_inicio}")
print(f"  Región: {region_filtro}")
print(f"  Producto: {producto_filtro}")
print(f"  Métricas: {metricas}")

In [ ]:
# Aplicar filtros dinámicamente
df_filtered = df_sales.filter(col("date") >= fecha_inicio)

if region_filtro != "All":
    df_filtered = df_filtered.filter(col("region") == region_filtro)
    
if producto_filtro != "All":
    df_filtered = df_filtered.filter(col("product") == producto_filtro)

print(f"📊 Registros después de filtrar: {df_filtered.count()}")
display(df_filtered)

## Ejercicio 3: Debugging y Logging

In [ ]:
import time
from functools import wraps

def time_it(func):
    """Decorator para medir tiempo de ejecución"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"⏱️  {func.__name__} ejecutado en {end - start:.2f} segundos")
        return result
    return wrapper

@time_it
def procesar_datos(df):
    """Simula procesamiento de datos"""
    result = df.groupBy("region", "product").agg(
        sum("amount").alias("total"),
        avg("amount").alias("promedio")
    ).collect()
    return result

# Ejecutar
resultado = procesar_datos(df_sales)
print(f"✅ Procesado {len(resultado)} grupos")

In [ ]:
# Función para detectar columnas con nulos
def find_null_columns(df):
    """Encuentra columnas con valores nulos"""
    null_counts = []
    
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            null_counts.append((col_name, null_count))
    
    return null_counts

# Probar
nulls = find_null_columns(df_sales)
if nulls:
    print("⚠️  Columnas con nulos:")
    for col_name, count in nulls:
        print(f"  - {col_name}: {count} nulos")
else:
    print("✅ No hay valores nulos")

In [ ]:
# Analizar plan de ejecución
df_complex = df_sales \
    .filter(col("amount") > 500) \
    .groupBy("region") \
    .agg(sum("amount").alias("total"))

print("📊 Plan de ejecución físico:")
df_complex.explain()

## Ejercicio 4: Guardar Resultados

In [ ]:
# Guardar resultados del análisis
table_name = "lab03_sales_analysis"

df_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable(table_name)

print(f"✅ Tabla creada: {table_name}")

In [ ]:
# Verificar tabla creada
print("📋 Verificación de tabla:")
print(f"  Tabla: {table_name}")

# Ver schema
df_read = spark.table(table_name)
print(f"  Registros: {df_read.count()}")
print(f"  Columnas: {', '.join(df_read.columns)}")

# Ver particiones (Delta Lake metadata)
spark.sql(f"DESCRIBE DETAIL {table_name}").select("partitionColumns").show(truncate=False)

## Resumen del Laboratorio

### Competencias Adquiridas:

**Magic Commands:**
- Uso de %md, %%sql, %sh, %fs para operaciones especializadas
- Integración de múltiples lenguajes en un mismo notebook

**Parametrización:**
- Implementación de widgets (text, dropdown, combobox, multiselect)
- Diseño de notebooks reutilizables con parámetros dinámicos

**Debugging y Optimización:**
- Decorators para medición de tiempo de ejecución
- Análisis de planes de ejecución con explain()
- Detección de valores nulos y validación de calidad de datos

**Persistencia:**
- Escritura de DataFrames particionados en formato Delta

### Próximos Pasos:
- **Lab 04**: Arquitectura Medallion completa (Bronze/Silver/Gold)
- **Lab 05**: Workflows de producción con error handling
- **Lab 06**: Integración de múltiples fuentes de datos